In [52]:
#ARI, NMI oder V-measure als Quantitative Maße für die Qualität deines ESM-Clusterings im Vergleich zu Canonical Clustern
#Werte liegen zwsichen 0 (schlechte überinstimmung) und 1 (perfekte Überinstimmung)

In [2]:

import os
import glob
import pandas as pd
from functools import reduce
from sklearn.metrics import v_measure_score


In [24]:
#angepasst auf finale datei
seq_regions = ['SEQ_H1', 'SEQ_H2', 'SEQ_L1', 'SEQ_L2', 'SEQ_L3']
cf_regions = ['CF_H1', 'CF_H2', 'CF_L1', 'CF_L2', 'CF_L3']
df = (
    pd.read_csv("data/ab_ag_scalop.tsv", sep="\t")
    .dropna(subset=seq_regions)
    .dropna(subset=cf_regions)
    .drop_duplicates(subset=seq_regions) 
)
antigen_counts = df["antigen_name"].value_counts() # Tabelle aus antigen_names und ihren Häufigkeiten in der Spalte antigen_name
df = df[df["antigen_name"].isin(antigen_counts[antigen_counts >= 5].index)] # Behält nur Zeilen, deren antigen_name mindestens 5-mal vorkommt
    

In [25]:


# CDR-Regionen definieren
cdrs = ["H1", "H2", "L1", "L2", "L3"]

# Mapping zwischen CF-Spalten und neuen HC-Spalten
cf_columns = [f"CF_{cdr}" for cdr in cdrs]
hc_columns = [f"HC_{cdr}" for cdr in cdrs]



# Lege leere HC-Spalten an
for hc_col in hc_columns:
    df[hc_col] = None

# Füge pro CDR-Typ die Clusterlabels hinzu
for cdr in cdrs:
    cluster_df = pd.read_csv(f"data/cdr_cluster_ESMC_tsvs/clusters_SEQ_{cdr}.tsv", sep="\t")

    # Mapping: pdb_id → cluster_label
    cluster_map = dict(zip(cluster_df["pdb"], cluster_df["cluster"]))

    # Schreibe ins Haupt-DataFrame
    df[f"HC_{cdr}"] = df["pdb"].map(cluster_map)

# Speichern als neue Datei
os.makedirs("data", exist_ok=True)
df.to_csv("data/ab_ag_hierarchical_canonical_forms_ESMC.csv", index=False)

In [5]:
# v measure ohne länge


# Dateien einlesen
df_true = df  # Original
df_pred = pd.read_csv("data/ab_ag_hierarchical_canonical_forms_ESMC.csv")  # Hierarchische Cluster

# CDRs, die verglichen werden sollen
cdrs = ["H1", "H2", "L1", "L2", "L3"]

print("V-Measure Ergebnisse:\n")

# Für jede Region CF vs HC vergleichen
for cdr in cdrs:
    true_labels = df_true[f"CF_{cdr}"]
    pred_labels = df_pred[f"HC_{cdr}"]
    
    v_score = v_measure_score(true_labels, pred_labels)
    print(f"CDR {cdr}: V-Measure = {v_score:.3f}")

V-Measure Ergebnisse:

CDR H1: V-Measure = 0.180
CDR H2: V-Measure = 0.268
CDR L1: V-Measure = 0.359
CDR L2: V-Measure = 0.000
CDR L3: V-Measure = 0.023


In [18]:
# einzelne CDR files zusammenführen


# 1) Input- und Output-Pfade
input_dir  = "data/cdr_cluster_ESMC_by_length_tsvs"
output_dir = "data/merged_clusters_per_region"
os.makedirs(output_dir, exist_ok=True)

# 2) Welche Regionen liegen in den Dateinamen?
#    (diese Liste entspricht den Teilen nach "SEQ_" in Deinen Dateinamen)
regions = ["H1", "H2", "L1", "L2", "L3"]

for region in regions:
    pattern = os.path.join(input_dir, f"clusters_SEQ_{region}_len*.tsv")
    files = glob.glob(pattern)
    if not files:
        print(f" Keine Dateien für SEQ_{region} gefunden ({pattern})")
        continue

    dfs = []
    for fn in files:
        df_length = pd.read_csv(fn, sep="\t", dtype={"pdb": str})
        # stell sicher, dass Hchain/Lchain existieren
        for c in ("Hchain","Lchain"):
            if c not in df_length.columns:
                df_length[c] = ""

        # neue Spalte im gewünschten Format
        col_new = f"HC_{region}"
        df_length[col_new] = (
            region
            + "_" + df_length["length"].astype(str)
            + "_" + df_length["cluster"].astype(str)
        )

        # nur die vier Spalten behalten
        dfs.append(df_length[["pdb","Hchain","Lchain",col_new]])

    # 3) alle Teildateien zusammenkleben
    merged = pd.concat(dfs, ignore_index=True)

    # 4) abspeichern
    out_fn = os.path.join(output_dir, f"merged_clusters_SEQ_{region}.tsv")
    merged.to_csv(out_fn, sep="\t", index=False)
    

In [19]:


# 1) Ordner mit Deinen per-Region-TSVs
in_dir  = "data/merged_clusters_per_region"
out_fn  = "data/ab_ag_clustered_all_CDRs_ESMC_by_length.csv"
os.makedirs(os.path.dirname(out_fn), exist_ok=True)

# 2) Liste der Regionen (im Dateinamen hinter „SEQ_“)
regions = ["H1","H2","L1","L2","L3"]

# 3) Für jede Region ein DataFrame einlesen
dfs = []
for r in regions:
    fn = os.path.join(in_dir, f"merged_clusters_SEQ_{r}.tsv")
    df_length = pd.read_csv(fn, sep="\t", dtype=str)
    # Spalte: SEQ_<r>_cluster_length
    # (steht schon so in Deinen Dateien, sonst hier umbenennen)
    dfs.append(df_length)

# 4) Alle auf pdb+Hchain+Lchain mergen (outer, damit nichts verloren geht)
merged = reduce(
    lambda left, right: left.merge(
        right,
        on=["pdb","Hchain","Lchain"],
        how="outer"
    ),
    dfs
)

# 5) Fehlendes auffüllen & optional Spalten-Reihenfolge
merged.fillna("", inplace=True)
cols = ["pdb","Hchain","Lchain"] + [f"HC_{r}" for r in regions]
merged = merged[cols]

# 6) Ergebnis speichern
merged.to_csv(out_fn, index=False)

In [20]:
# v measure mit länge

# Dateien einlesen
df_true = df  # Original
df_pred = pd.read_csv("data/ab_ag_clustered_all_CDRs_ESMC_by_length.csv")  # Hierarchische Cluster

print(df_pred.columns.tolist())
# CDRs, die verglichen werden sollen
cdr = ["H1", "H2", "L1", "L2", "L3"]

print("V-Measure Ergebnisse:\n")

# Für jede Region CF vs HC vergleichen
for cdr in cdrs:
    true_labels = df_true[f"CF_{cdr}"]
    pred_labels = df_pred[f"HC_{cdr}"]
    
    v_score = v_measure_score(true_labels, pred_labels)
    print(f"CDR {cdr}: V-Measure = {v_score:.3f}")

['pdb', 'Hchain', 'Lchain', 'HC_H1', 'HC_H2', 'HC_L1', 'HC_L2', 'HC_L3']
V-Measure Ergebnisse:

CDR H1: V-Measure = 0.026
CDR H2: V-Measure = 0.032
CDR L1: V-Measure = 0.087
CDR L2: V-Measure = 0.000
CDR L3: V-Measure = 0.045


In [23]:
# einzelne CDR files zusammenführen pre defined
# 1) Input- und Output-Pfade
input_dir  = "data/cdr_cluster_ESMC_by_length_pre_defined_tsvs"
output_dir = "data/merged_clusters_per_region_pre defined"
os.makedirs(output_dir, exist_ok=True)

# 2) Welche Regionen liegen in den Dateinamen?
#    (diese Liste entspricht den Teilen nach "SEQ_" in Deinen Dateinamen)
regions = ["H1", "H2", "L1", "L2", "L3"]

for region in regions:
    pattern = os.path.join(input_dir, f"cdr_clusters_ESMCSEQ_{region}_len*.tsv")
    files = glob.glob(pattern)
    if not files:
        print(f" Keine Dateien für SEQ_{region} gefunden ({pattern})")
        continue

    dfs = []
    for fn in files:
        df_length_pre = pd.read_csv(fn, sep="\t", dtype={"pdb": str})
        # stell sicher, dass Hchain/Lchain existieren
        for c in ("Hchain","Lchain"):
            if c not in df_length_pre.columns:
                df_length_pre[c] = ""

        # neue Spalte im gewünschten Format
        col_new = f"HC_{region}"
        df_length_pre[col_new] = (
            region
            + "_" + df_length_pre["length"].astype(str)
            + "_" + df_length_pre["cluster"].astype(str)
        )

        # nur die vier Spalten behalten
        dfs.append(df_length_pre[["pdb","Hchain","Lchain",col_new]])

    # 3) alle Teildateien zusammenkleben
    merged = pd.concat(dfs, ignore_index=True)

    # 4) abspeichern
    out_fn = os.path.join(output_dir, f"merged_clusters_SEQ_{region}.tsv")
    merged.to_csv(out_fn, sep="\t", index=False)

In [ ]:


# 1) Ordner mit Deinen per-Region-TSVs
in_dir  = "data/merged_clusters_per_region_pre defined"
out_fn  = "data/ab_ag_clustered_all_CDRs_ESMC_by_length_pre_deefined.csv"
os.makedirs(os.path.dirname(out_fn), exist_ok=True)

# 2) Liste der Regionen (im Dateinamen hinter „SEQ_“)
regions = ["H1","H2","L1","L2","L3"]

# 3) Für jede Region ein DataFrame einlesen
dfs = []
for r in regions:
    fn = os.path.join(in_dir, f"merged_clusters_SEQ_{r}.tsv")
    df = pd.read_csv(fn, sep="\t", dtype=str)
    # Spalte: SEQ_<r>_cluster_length
    # (steht schon so in Deinen Dateien, sonst hier umbenennen)
    dfs.append(df)

# 4) Alle auf pdb+Hchain+Lchain mergen (outer, damit nichts verloren geht)
merged = reduce(
    lambda left, right: left.merge(
        right,
        on=["pdb","Hchain","Lchain"],
        how="outer"
    ),
    dfs
)

# 5) Fehlendes auffüllen & optional Spalten-Reihenfolge
merged.fillna("", inplace=True)
cols = ["pdb","Hchain","Lchain"] + [f"HC_{r}" for r in regions]
merged = merged[cols]

# 6) Ergebnis speichern
merged.to_csv(out_fn, index=False)

In [ ]:
# v measure mit länge


# Dateien einlesen
df_true = df  # Original
df_pred = pd.read_csv("data/ab_ag_clustered_all_CDRs_ESMC_by_length_pre_deefined.csv")  # Hierarchische Cluster

print(df_pred.columns.tolist())
# CDRs, die verglichen werden sollen
cdr = ["H1", "H2", "L1", "L2", "L3"]

print("V-Measure Ergebnisse:\n")

# Für jede Region CF vs HC vergleichen
for cdr in cdrs:
    true_labels = df_true[f"CF_{cdr}"]
    pred_labels = df_pred[f"HC_{cdr}"]
    
    v_score = v_measure_score(true_labels, pred_labels)
    print(f"CDR {cdr}: V-Measure = {v_score:.3f}")

['pdb', 'Hchain', 'Lchain', 'HC_H1', 'HC_H2', 'HC_L1', 'HC_L2', 'HC_L3']
V-Measure Ergebnisse:

CDR H1: V-Measure = 0.026
CDR H2: V-Measure = 0.024
CDR L1: V-Measure = 0.044
CDR L2: V-Measure = 0.000
CDR L3: V-Measure = 0.034
